# Лекция 3. Онтология и вывод: демонстрация

Курс «От интеллектуального поиска к интеллектуальным агентам», ВМК МГУ. О. М. Атаева, А. П. Халов.

Блокнот продолжает задачу редактора. Онтология лекции 2 (`data/review.owl`) пересматривается по вопросам редактора,
машина вывода относит статьи к выводимому классу и объясняет выводы, формы SHACL проверяют данные в замкнутом мире.

| Шаг | Что делаем |
|---|---|
| 0 | Загружаем онтологию лекции 2 |
| 1 | Пять вопросов редактора и их средства |
| 2 | Пересмотр: два выводимых класса и свойство `срок` |
| 3 | Машина вывода: класс пуст, замыкание мира, четыре кандидата |
| 4 | Объяснения: почему P3, P6 и P4 не готовы |
| 5 | Неудовлетворимый класс в схеме |
| 6 | Граф форм SHACL |
| 7 | Проверка данных: отчёт |
| 8 | Строка с ошибкой: форма и машина вывода рядом |
| 9 | Ответ редактора: два средства вместе |
| 10 | Агент с инструментом проверки данных |
| 11 | Вариант по математике |

Для шагов 3–5, 8 и 10 нужна Java. Без неё блокнот печатает сохранённую запись, проверка SHACL работает всегда.

## Шаг 0. Онтология лекции 2

In [1]:
import pathlib, datetime, warnings, io, contextlib, os, json, re, shutil
import pandas as pd
import rdflib
from pyshacl import validate
warnings.filterwarnings("ignore")
from owlready2 import *
from rdflib import Literal, RDF, XSD       # после owlready2: у него своё имя Namespace

DATA = pathlib.Path("data")
HAS_JAVA = shutil.which("java") is not None
world = World()
onto = world.get_ontology((DATA / "review.owl").resolve().as_uri()).load()
R = rdflib.Namespace("http://example.org/review.owl#")
SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
print("классов:", len(list(onto.classes())), "| объектных свойств:", len(list(onto.object_properties())),
      "| индивидов:", len(list(onto.individuals())), "| правил:", len(list(onto.rules())), "| Java:", HAS_JAVA)
print("классы:", ", ".join(c.name for c in onto.classes()))

классов: 12 | объектных свойств: 10 | индивидов: 69 | правил: 1 | Java: True
классы: Положение, Рецензия, Статья, Рецензент, Организация, Требование, Утверждение, Доказательство, Объект, Раздел, Термин, БазовоеПоложение


## Модель: настройка

Живые ячейки шага 10 (агент с двумя верификаторами) обращаются к языковой модели через OpenAI-совместимый API. Три значения
задаются здесь явно: ключ, адрес сервера и имя модели. Ячейка проверяет, отвечает ли сервер. Если нет, ключ
сбрасывается, и шаг 10 печатает записанный протокол, остальные шаги от модели не зависят.

In [2]:
import os

# Вариант 1. Локальная Ollama: сервер на этой машине, модель с вызовом функций (qwen-agent собрана из Modelfile
#            командой ollama create qwen-agent -f Modelfile, подойдёт и qwen2.5:3b). Ключ для Ollama любая непустая строка.
LLM_API_KEY  = "ollama"
LLM_BASE_URL = "http://127.0.0.1:11434/v1"
LLM_MODEL    = "qwen-agent"

# Вариант 2. Облачная модель: подставить свои значения вместо трёх строк выше, например
#   LLM_API_KEY = "sk-...";  LLM_BASE_URL = "https://api.openai.com/v1";  LLM_MODEL = "gpt-4o-mini"
# Вариант 3. Без модели: LLM_API_KEY = "". Шаг 10 печатает записанный протокол.

os.environ["LLM_BASE_URL"], os.environ["LLM_MODEL"] = LLM_BASE_URL, LLM_MODEL
if LLM_BASE_URL.startswith(("http://127.0.0.1", "http://localhost")):
    for k in ("ALL_PROXY", "all_proxy", "HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy"):
        os.environ.pop(k, None)    # локальный сервер: системный прокси этому блокноту не нужен

# Проверка сервера. Ключ попадает в окружение только если сервер ответил.
os.environ["LLM_API_KEY"] = ""
if LLM_API_KEY:
    try:
        from openai import OpenAI
    except ModuleNotFoundError:
        OpenAI = None
        print("пакет openai не установлен в ядре блокнота: выполните %pip install openai и перезапустите ядро. Живые ячейки идут по записи")
    try:
        if OpenAI is None:
            raise RuntimeError("нет пакета openai")
        names = [m.id for m in OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL, timeout=5, max_retries=0).models.list()]
        os.environ["LLM_API_KEY"] = LLM_API_KEY
        if names and LLM_MODEL not in names and LLM_MODEL + ":latest" not in names:
            print(f"сервер отвечает, модели {LLM_MODEL} в его списке нет (есть: {', '.join(names[:6])}), вызовы могут не пройти")
        else:
            print(f"модель: {LLM_MODEL} через {LLM_BASE_URL}")
    except Exception as e:
        if OpenAI is not None:
            print(f"сервер {LLM_BASE_URL} не отвечает ({type(e).__name__}): шаг 10 идёт по записи")
else:
    print("модель не задана: шаг 10 идёт по записи")

сервер http://127.0.0.1:11434/v1 не отвечает (APIConnectionError): шаг 10 идёт по записи


## Шаг 1. Пять вопросов редактора и их средства

Вопросы исследователя из задания лекции 1 становятся компетентностными вопросами к онтологии.
У каждого есть ожидаемый ответ, источник и средство, которым онтология на него отвечает.

In [3]:
ВОПРОСЫ = [
    ("Какие статьи готовы к решению редколлегии?",        "выводимый класс + проверка данных", "P1, P2, P7"),
    ("На какие положения опирается требование 6.5?",      "транзитивное свойство",             "14 положений"),
    ("У каких статей рецензент из организации авторов?",  "правило или проверка",              "P3"),
    ("Кто рецензировал статьи организации МГУ?",          "запрос (лекция 4)",                 "Rv1, Rv2, Rv3, Rv6"),
    ("Есть ли норма о сроке без слова «срок»?",           "поиск по смыслу (лекция 6)",        "требование 6.6"),
]
pd.DataFrame(ВОПРОСЫ, columns=["вопрос", "средство", "ожидаемый ответ"])

                                             вопрос                           средство     ожидаемый ответ
0        Какие статьи готовы к решению редколлегии?  выводимый класс + проверка данных          P1, P2, P7
1      На какие положения опирается требование 6.5?              транзитивное свойство        14 положений
2  У каких статей рецензент из организации авторов?               правило или проверка                  P3
3          Кто рецензировал статьи организации МГУ?                  запрос (лекция 4)  Rv1, Rv2, Rv3, Rv6
4           Есть ли норма о сроке без слова «срок»?         поиск по смыслу (лекция 6)      требование 6.6


Вопрос 1 требует классов, которых в онтологии лекции 2 нет: «сданная рецензия» и «готовая к решению статья».
Отсюда пересмотр. Таблица «вопрос — класс» заполняется до записи.

In [4]:
ТАБЛИЦА = {
    1: ["Статья", "Рецензия", "СданнаяРецензия", "ГотоваяКРешениюСтатья", "имеет_рецензию", "ответ_авторов", "нарушает", "срок"],
    2: ["Положение", "Требование", "зависит_от"],
    3: ["Статья", "Рецензия", "Рецензент", "Организация", "из_организации", "организация_авторов"],
}
используется = {x for v in ТАБЛИЦА.values() for x in v}
без_вопроса = [c.name for c in onto.classes() if c.name not in используется]
print("классы без вопроса (кандидаты на удаление):", без_вопроса)

классы без вопроса (кандидаты на удаление): ['Утверждение', 'Доказательство', 'Объект', 'Раздел', 'Термин', 'БазовоеПоложение']


## Шаг 2. Пересмотр онтологии

Три записи: свойство `срок` (назначено плюс 21 день по положению), выводимый класс «сданная рецензия»,
выводимый класс «готовая к решению статья» по вопросу 1. Выражение класса читается как три требования положения:
две сданные рецензии (6.1), у всех рецензий ответ авторов (6.5), нет нарушений (6.2). Срок (6.6) требует
сравнения дат и в выражение класса не входит.

In [5]:
with onto:
    class срок(DataProperty, FunctionalProperty):
        domain = [onto.Рецензия]; range = [datetime.date]
    for r in onto.Рецензия.instances():
        if r.назначено:
            r.срок = r.назначено + datetime.timedelta(days=21)

    class СданнаяРецензия(onto.Рецензия):
        equivalent_to = [onto.Рецензия & onto.сдано.min(1, datetime.date)]

    class ГотоваяКРешениюСтатья(onto.Статья):
        equivalent_to = [onto.Статья
                         & onto.имеет_рецензию.min(2, СданнаяРецензия)
                         & onto.имеет_рецензию.only(onto.Рецензия & onto.ответ_авторов.value(True))
                         & Not(onto.нарушает.some(onto.Требование))]

for c in (СданнаяРецензия, ГотоваяКРешениюСтатья):
    print(c.name, "≡", str(c.equivalent_to[0]).replace("review.", ""))
print("срок у рецензии P5/Rv3:", onto["рец_P5_Rv3"].назначено, "→", onto["рец_P5_Rv3"].срок, "| сдано:", onto["рец_P5_Rv3"].сдано)

СданнаяРецензия ≡ Рецензия & сдано.min(1, <class 'datetime.date'>)
ГотоваяКРешениюСтатья ≡ Статья & имеет_рецензию.min(2, СданнаяРецензия) & имеет_рецензию.only(Рецензия & ответ_авторов.value(True)) & Not(нарушает.some(Требование))
срок у рецензии P5/Rv3: 2026-09-03 → 2026-09-24 | сдано: 2026-09-30


## Шаг 3. Машина вывода: открытый мир и замыкание

Первый запуск: класс «сданная рецензия» наполняется, класс «готовая к решению статья» остаётся пустым.
Условия «все рецензии с ответом» и «нет нарушений» в открытом мире недоказуемы по записи.
После замыкания мира для статей машина относит к классу четыре статьи.

In [6]:
def run_reasoner(explain=False):
    """Запускает Pellet. Возвращает (успех, текст сообщения)."""
    if not HAS_JAVA:
        return None, "Java не найдена"
    buf = io.StringIO()
    try:
        with contextlib.redirect_stderr(buf), contextlib.redirect_stdout(buf):
            with onto:
                sync_reasoner_pellet(world, infer_property_values=True, debug=2 if explain else 0)
        return True, ""
    except OwlReadyInconsistentOntologyError as e:
        text = str(e)
        return False, text[text.find("Explanation"):] if "Explanation" in text else text

ok, _ = run_reasoner()
if ok is None:
    print('''[запись] вывод машины без Java
СданнаяРецензия: 13 индивидов
ГотоваяКРешениюСтатья до замыкания: []
ГотоваяКРешениюСтатья после замыкания: ['P1', 'P2', 'P5', 'P7']''')
else:
    print("СданнаяРецензия:", len(list(СданнаяРецензия.instances())), "индивидов")
    print("ГотоваяКРешениюСтатья до замыкания:", sorted(x.name for x in ГотоваяКРешениюСтатья.instances()))
    with onto:
        for a in onto.Статья.instances():
            close_world(a, Properties=[onto.имеет_рецензию, onto.нарушает])
    run_reasoner()
    ГОТОВЫ = sorted(x.name for x in ГотоваяКРешениюСтатья.instances())
    print("ГотоваяКРешениюСтатья после замыкания:", ГОТОВЫ)

СданнаяРецензия: 13 индивидов
ГотоваяКРешениюСтатья до замыкания: []
ГотоваяКРешениюСтатья после замыкания: ['P1', 'P2', 'P5', 'P7']


Четыре кандидата: P1, P2, P7 и P5. У P5 рецензия сдана на 27-й день, но срока в выражении класса нет.
P3 исключена правилом 6.2, P6 ответом авторов, P4 и P8 числом сданных рецензий.

## Шаг 4. Объяснения

Для каждой статьи, которая не попала в класс, объяснение собирается из фактов журнала и аксиом определения.

In [7]:
def почему_не_готова(name):
    a = onto[name]
    reviews = [r for r in onto.Рецензия.instances() if r.на_статью == a]
    lines = []
    for t in a.нарушает:
        rv = next(r for r in reviews if r.рецензент and r.рецензент.из_организации == a.организация_авторов)
        lines.append(f"{name} нарушает {t.name} ({t.label[0]})  [выведено правилом 6.2 из фактов: "
                     f"{rv.name} рецензент {rv.рецензент.name}, {rv.рецензент.name} из_организации "
                     f"{rv.рецензент.из_организации[0].name}, {name} организация_авторов {a.организация_авторов[0].name}]")
        lines.append("  аксиома: ГотоваяКРешениюСтатья ⊑ ¬∃нарушает.Требование")
    без_ответа = [r.name for r in reviews if r.сдано and not r.ответ_авторов]
    if без_ответа:
        lines.append(f"ответ авторов не записан: {', '.join(без_ответа)}")
        lines.append("  аксиома: ГотоваяКРешениюСтатья ⊑ ∀имеет_рецензию.(ответ_авторов = true)")
    сданных = sum(1 for r in reviews if r.сдано)
    if сданных < 2:
        lines.append(f"сданных рецензий {сданных}, требуется не меньше двух")
        lines.append("  аксиома: ГотоваяКРешениюСтатья ⊑ ≥2 имеет_рецензию.СданнаяРецензия")
    return lines or ["условия выполнены: статья в классе"]

for name in ("P3", "P6", "P4", "P1"):
    print(f"почему {name}:")
    for l in почему_не_готова(name):
        print("  " + l)

почему P3:
  P3 нарушает R62 (6.2 Независимость рецензента)  [выведено правилом 6.2 из фактов: рец_P3_Rv4 рецензент Rv4, Rv4 из_организации ФИЦ_ИУ, P3 организация_авторов ФИЦ_ИУ]
    аксиома: ГотоваяКРешениюСтатья ⊑ ¬∃нарушает.Требование
почему P6:
  ответ авторов не записан: рец_P6_Rv4, рец_P6_Rv5
    аксиома: ГотоваяКРешениюСтатья ⊑ ∀имеет_рецензию.(ответ_авторов = true)
почему P4:
  сданных рецензий 1, требуется не меньше двух
    аксиома: ГотоваяКРешениюСтатья ⊑ ≥2 имеет_рецензию.СданнаяРецензия
почему P1:
  условия выполнены: статья в классе


## Шаг 5. Неудовлетворимый класс

Проверка схемы без данных: класс, у которого не может быть экземпляров. Классы «Статья» и «Положение» объявлены
непересекающимися в лекции 2, их общий подкласс пуст по определению. Машина сообщает об этом эквивалентностью классу `Nothing`.

In [8]:
with onto:
    class СтатьяПоложение(onto.Статья, onto.Положение): pass
if HAS_JAVA:
    run_reasoner()
    print("СтатьяПоложение.equivalent_to =", list(СтатьяПоложение.equivalent_to))
    print("неудовлетворимые классы:", [c.name for c in onto.classes() if Nothing in c.equivalent_to])
    destroy_entity(СтатьяПоложение)
else:
    print("[запись] СтатьяПоложение.equivalent_to = [owl.Nothing]")

СтатьяПоложение.equivalent_to = [owl.Nothing]
неудовлетворимые классы: ['СтатьяПоложение']


## Шаг 6. Граф форм SHACL

Формы записаны в `data/shapes.ttl`: пять требований положения и две проверки записи. Граф данных берётся из файла
`review.owl` как он сохранён в лекции 2, с добавленным свойством `срок`. Выведенные машиной факты в него не входят.

In [9]:
print((DATA / "shapes.ttl").read_text(encoding="utf-8"))

# Формы SHACL для положения о статьях. Лекция 3.
# Граф данных: review.owl лекции 2 плюс свойство :срок, вычисленное при загрузке (назначено + 21 день).
@prefix sh:  <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix :    <http://example.org/review.owl#> .
@prefix shp: <http://example.org/shapes#> .

# 6.1. У статьи не меньше двух сданных рецензий. В файле записано только :на_статью, поэтому путь обратный.
shp:Статья a sh:NodeShape ;
    sh:targetClass :Статья ;
    sh:property [
        sh:path [ sh:inversePath :на_статью ] ;
        sh:qualifiedValueShape [ sh:property [ sh:path :сдано ; sh:minCount 1 ] ] ;
        sh:qualifiedMinCount 2 ;
        sh:message "6.1: сданных рецензий меньше двух" ] .

shp:Рецензия a sh:NodeShape ;
    sh:targetClass :Рецензия ;
    # запись: у рецензии ровно одна статья, и это статья
    sh:property [
        sh:path :на_статью ;
        sh:class :Статья ;
        sh:minCount 1 ; sh:maxCount 1 ;
        sh:message

In [10]:
data = rdflib.Graph(); data.parse(str(DATA / "review.owl"), format="xml")
for s, o in list(data.subject_objects(R["назначено"])):
    data.add((s, R["срок"], Literal(datetime.date.fromisoformat(str(o)) + datetime.timedelta(days=21), datatype=XSD.date)))
shapes = rdflib.Graph(); shapes.parse(str(DATA / "shapes.ttl"), format="turtle")
print("троек данных:", len(data), "| троек форм:", len(shapes))
print("троек имеет_рецензию в файле:", len(list(data.triples((None, R["имеет_рецензию"], None)))),
      "| на_статью:", len(list(data.triples((None, R["на_статью"], None)))))

троек данных: 579 | троек форм: 45
троек имеет_рецензию в файле: 0 | на_статью: 14


Свойство `имеет_рецензию` объявлено обратным к `на_статью` и в лекции 2 выводилось машиной. В файле его нет.
Поэтому форма для 6.1 записана через `sh:inversePath`: проверка видит только записанное.

## Шаг 7. Проверка данных

In [11]:
def проверить(g, shapes=shapes):
    ok, report, text = validate(g, shacl_graph=shapes, inference="none")
    rows = []
    for r in report.subjects(RDF.type, SH.ValidationResult):
        path = report.value(r, SH.resultPath)
        inv = report.value(path, SH.inversePath) if path is not None else None
        rows.append({"узел": str(report.value(r, SH.focusNode)).split("#")[-1],
                     "свойство": ("← " + str(inv).split("#")[-1]) if inv is not None else (str(path or "").split("#")[-1] or "—"),
                     "сообщение": str(report.value(r, SH.resultMessage)),
                     "строгость": str(report.value(r, SH.resultSeverity)).split("#")[-1]})
    return ok, pd.DataFrame(rows).sort_values(["строгость", "узел"], ascending=[False, True]).reset_index(drop=True), text

ok, отчёт, text = проверить(data)
print("Conforms:", ok, "| нарушений:", (отчёт["строгость"] == "Violation").sum(), "| предупреждений:", (отчёт["строгость"] == "Warning").sum())
отчёт

Conforms: False | нарушений: 4 | предупреждений: 3
         узел       свойство                              сообщение  строгость
0  рец_P6_Rv4  ответ_авторов          6.5: ответ авторов не записан    Warning
1  рец_P6_Rv5  ответ_авторов          6.5: ответ авторов не записан    Warning
2  рец_P8_Rv5  ответ_авторов          6.5: ответ авторов не записан    Warning
3          P4    ← на_статью      6.1: сданных рецензий меньше двух  Violation
4          P8    ← на_статью      6.1: сданных рецензий меньше двух  Violation
5  рец_P3_Rv4              —  6.2: рецензент из организации авторов  Violation
6  рец_P5_Rv3          сдано        6.6: рецензия сдана позже срока  Violation


Три требования, которые лекция 2 оставила без проверки, получили сообщения: 6.1 у P4 и P8, 6.6 у рецензии P5.
Требование 6.2 найдено и правилом, и формой. Предупреждения по 6.5 приём данных не останавливают.
Так выглядит отчёт в том виде, как его печатает pySHACL:

In [12]:
print(text[:1200])

Validation Report
Conforms: False
Results (7):
Constraint Violation in QualifiedValueShapeConstraintComponent (http://www.w3.org/ns/shacl#QualifiedMinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:message Literal("6.1: сданных рецензий меньше двух") ; sh:path [ sh:inversePath :на_статью ] ; sh:qualifiedMinCount Literal("2", datatype=xsd:integer) ; sh:qualifiedValueShape [ sh:property [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:path :сдано ] ] ]
	Focus Node: :P8
	Result Path: [ sh:inversePath :на_статью ]
	Message: 6.1: сданных рецензий меньше двух
Constraint Violation in QualifiedValueShapeConstraintComponent (http://www.w3.org/ns/shacl#QualifiedMinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:message Literal("6.1: сданных рецензий меньше двух") ; sh:path [ sh:inversePath :на_статью ] ; sh:qualifiedMinCount Literal("2", datatype=xsd:integer) ; sh:qualifiedValueShape [ sh:property [ sh:minCount Literal("1", datatype=xsd:integer)

## Шаг 8. Строка с ошибкой: два средства рядом

В поле «рецензент» новой рецензии записана организация. Форма сообщает о неверном классе значения.
Машина вывода выводит тип из области значений и находит противоречие с дизъюнктностью.

In [13]:
bad = data + rdflib.Graph()
bad.add((R["рец_P9_ошибка"], RDF.type, R["Рецензия"]))
bad.add((R["рец_P9_ошибка"], R["на_статью"], R["P8"]))
bad.add((R["рец_P9_ошибка"], R["рецензент"], R["ФИЦ_ИУ"]))
ok, отчёт_bad, _ = проверить(bad)
print("ПРОВЕРКА ПО ФОРМЕ")
print(отчёт_bad[отчёт_bad["узел"] == "рец_P9_ошибка"].to_string(index=False))

print("\nМАШИНА ВЫВОДА")
if HAS_JAVA:
    # свежая копия онтологии лекции 2: без замыкания мира объяснение короче
    w2 = World(); o2 = w2.get_ontology((DATA / "review.owl").resolve().as_uri()).load()
    with o2:
        ошибка = o2.Рецензия("рец_P9_ошибка"); ошибка.на_статью = o2.P8; ошибка.рецензент = o2["ФИЦ_ИУ"]
    try:
        with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
            with o2: sync_reasoner_pellet(w2, infer_property_values=True, debug=2)
        print("непротиворечива: True")
    except OwlReadyInconsistentOntologyError as e:
        msg = str(e); msg = msg[msg.find("Explanation"):]
        print("непротиворечива: False")
        print(re.sub(r"\n\s*\n", "\n", msg.replace("review.", "")).strip()[:600])
else:
    print('''[запись] непротиворечива: False
Explanation(s):
1) DisjointClasses(... Организация ... Рецензент ...)
   рец_P9_ошибка рецензент ФИЦ_ИУ
   рецензент range Рецензент
   ФИЦ_ИУ type Организация''')

ПРОВЕРКА ПО ФОРМЕ
         узел      свойство                         сообщение строгость
рец_P9_ошибка ответ_авторов     6.5: ответ авторов не записан   Warning
рец_P9_ошибка     рецензент рецензентом должен быть рецензент Violation

МАШИНА ВЫВОДА
непротиворечива: False
Explanation(s): 
1)   DisjointClasses(Доказательство
                     Организация
                     Положение
                     Рецензент
                     Рецензия
                     Статья
                     Утверждение)
     рец_P9_ошибка рецензент ФИЦ_ИУ
     рецензент range Рецензент
     ФИЦ_ИУ type Организация


Ошибка одна, сообщения два. Форма называет узел, свойство и значение. Машина называет четыре аксиомы и факта.
Проверка по форме стоит в проекте первой: она дешевле и говорит, какое поле исправить.

## Шаг 9. Ответ редактора: два средства вместе

In [14]:
ГОТОВЫ = ГОТОВЫ if HAS_JAVA else ["P1", "P2", "P5", "P7"]
наруш = отчёт[отчёт["строгость"] == "Violation"]
rows = []
for name in sorted(a.name for a in onto.Статья.instances()):
    reviews = [r for r in onto.Рецензия.instances() if r.на_статью == onto[name]]
    узлы = [name] + [r.name for r in reviews]
    причины = [m.split(":")[0] for m in наруш[наруш["узел"].isin(узлы)]["сообщение"]]
    пред = [m.split(":")[0] for m in отчёт[(отчёт["строгость"] == "Warning") & отчёт["узел"].isin(узлы)]["сообщение"]]
    машина = "готова" if name in ГОТОВЫ else ("нарушает 6.2" if onto[name].нарушает else "—")
    rows.append({"статья": name, "машина вывода": машина,
                 "проверка данных": ", ".join(sorted(set(причины + пред))) or "нарушений нет",
                 "итог": "выносить" if name in ГОТОВЫ and not причины and not пред else "нет"})
pd.DataFrame(rows)

  статья машина вывода проверка данных      итог
0     P1        готова   нарушений нет  выносить
1     P2        готова   нарушений нет  выносить
2     P3  нарушает 6.2             6.2       нет
3     P4             —             6.1       нет
4     P5        готова             6.6       нет
5     P6             —             6.5       нет
6     P7        готова   нарушений нет  выносить
7     P8             —        6.1, 6.5       нет


Готовы P1, P2, P7: тот же ответ, что дали четыре правила в коде на лекции 1. Теперь знание о требованиях записано
в онтологии и в формах, и каждое «нет» имеет пункт положения и строку отчёта.

| Механизм | Что дал |
|---|---|
| компетентностные вопросы | два новых класса и свойство, один класс без вопроса |
| машина вывода, классификация | четыре кандидата после замыкания мира, объяснение для каждой статьи вне класса |
| машина вывода, схема | неудовлетворимый класс найден без данных |
| SHACL | 6.1 у P4 и P8, 6.6 у P5, 6.2 у P3, предупреждения 6.5, строка с ошибкой |
| оба вместе | ответ редактора P1, P2, P7 |

## Шаг 10. Агент с инструментом проверки данных

Агент лекции 2 получает два инструмента: `validate_data` возвращает отчёт по формам, `ready_articles` возвращает
выводимый класс после замыкания. К пометкам ответа добавляется «не прошло проверку» с пунктом положения.
Без ключа доступа к модели печатается протокол: инструменты вызываются на самом деле, текст ответа записан.

In [15]:
from agent_core import run_agent, openai_llm, scripted_llm

def check_consistency():
    ok, msg = run_reasoner(explain=True) if HAS_JAVA else (True, "")
    return {"непротиворечива": bool(ok)} if ok else {"непротиворечива": False, "объяснение": msg[:300]}

def validate_data():
    ok, rep, _ = проверить(data)
    return {"прошло": bool(ok), "нарушения": [
        {"узел": r["узел"], "требование": r["сообщение"].split(":")[0], "сообщение": r["сообщение"].split(": ", 1)[-1],
         "строгость": r["строгость"]} for _, r in rep.iterrows()]}

def ready_articles():
    return {"готовы": ГОТОВЫ, "основание": "класс ГотоваяКРешениюСтатья после замыкания мира"}

def infer(article):
    a = onto[article]
    if a is None:
        return {"ошибка": f"статья {article} не найдена"}
    return {"статья": article, "выведено": [{"факт": f"{article} нарушает {t.name} ({t.label[0]})", "правило": "правило 6.2 (SWRL)"} for t in a.нарушает]}

TOOLS = [
  {"type": "function", "function": {"name": "check_consistency", "description": "Проверка непротиворечивости записи.", "parameters": {"type": "object", "properties": {}}}},
  {"type": "function", "function": {"name": "validate_data", "description": "Проверка данных по формам положения: отчёт с нарушениями и пунктами требований.", "parameters": {"type": "object", "properties": {}}}},
  {"type": "function", "function": {"name": "ready_articles", "description": "Статьи, которые машина вывода относит к готовым к решению.", "parameters": {"type": "object", "properties": {}}}},
  {"type": "function", "function": {"name": "infer", "description": "Факты о статье, выведенные машиной вывода, с указанием правила.", "parameters": {"type": "object", "properties": {"article": {"type": "string"}}, "required": ["article"]}}},
]
IMPL = {"check_consistency": check_consistency, "validate_data": validate_data, "ready_articles": ready_articles, "infer": infer}
SYSTEM = ("Ты ассистент редактора сборника. Отвечай по-русски. Сначала вызови check_consistency, затем validate_data. "
          "Статья считается готовой, если она есть в ready_articles и у неё и её рецензий нет нарушений в отчёте. "
          "Отвечай только по результатам инструментов. Помечай части ответа: [выведено: ...], [найдено: ...], "
          "[не прошло проверку: пункт], [не подтверждено].")
QUESTION = "Какие статьи можно выносить на решение редколлегии и почему остальные нельзя?"
PLAN = [("check_consistency", {}), ("validate_data", {}), ("ready_articles", {})]
RECORD = ("[запись ответа модели]\n"
          "Запись непротиворечива [найдено: check_consistency]. Готовы к решению P1, P2, P7 [выведено: класс "
          "ГотоваяКРешениюСтатья] [проверено: validate_data]. P5 выведена как готовая, но её рецензия сдана позже срока "
          "[не прошло проверку: 6.6]. P3 нарушает 6.2 [выведено: правило 6.2] [не прошло проверку: 6.2]. P4 и P8: сданных "
          "рецензий меньше двух [не прошло проверку: 6.1]. P6: ответ авторов не записан [не прошло проверку: 6.5].")
llm = openai_llm if os.environ.get("LLM_API_KEY") else scripted_llm(PLAN, RECORD)
_ = run_agent(QUESTION, llm, SYSTEM, TOOLS, IMPL)

[1] модель вызвала check_consistency({})
    <- {"непротиворечива": true}
[2] модель вызвала validate_data({})
    <- {"прошло": false, "нарушения": [{"узел": "рец_P6_Rv4", "требование": "6.5", "сообщение": "ответ авторов не записан", "строгость": "Warning"}, {"узел": "рец_P6_R…
[3] модель вызвала ready_articles({})
    <- {"готовы": ["P1", "P2", "P5", "P7"], "основание": "класс ГотоваяКРешениюСтатья после замыкания мира"}
[4] ответ модели:
[запись ответа модели]
Запись непротиворечива [найдено: check_consistency]. Готовы к решению P1, P2, P7 [выведено: класс ГотоваяКРешениюСтатья] [проверено: validate_data]. P5 выведена как готовая, но её рецензия сдана позже срока [не прошло проверку: 6.6]. P3 нарушает 6.2 [выведено: правило 6.2] [не прошло проверку: 6.2]. P4 и P8: сданных рецензий меньше двух [не прошло проверку: 6.1]. P6: ответ авторов не записан [не прошло проверку: 6.5].


Ответ редактора собран агентом из двух верификаторов. Критерий приёмки записан формой: статья принимается,
когда данные о ней прошли проверку. Каждая часть ответа несёт пометку и пункт положения.

## Шаг 11. Вариант по математике

Тот же скелет на утверждениях статьи P2 и корпусе планиметрии: выводимый класс «доказанная теорема» по вердикту
внешней проверки и форма «у каждого утверждения есть доказательство».

In [16]:
claims = pd.read_csv(DATA / "paper_P2_claims.csv", dtype=str).fillna("")
mw = World(); m = mw.get_ontology("http://example.org/math.owl")
with m:
    class Утверждение(Thing): pass
    class Теорема(Утверждение): pass
    class Доказательство(Thing): pass
    class доказывается(ObjectProperty): domain = [Утверждение]; range = [Доказательство]
    class вердикт(DataProperty, FunctionalProperty): domain = [Доказательство]; range = [str]
    class ДоказанноеУтверждение(Утверждение):
        equivalent_to = [Утверждение & доказывается.some(Доказательство & вердикт.value("принято"))]
    for _, c in claims.iterrows():
        u = Утверждение(c["id"]); d = Доказательство("Док_" + c["id"]); d.вердикт = c["вердикт_lean"].split(":")[0]; u.доказывается = [d]
    Утверждение("C4")                       # утверждение без доказательства, для формы
if HAS_JAVA:
    with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
        with m: sync_reasoner_pellet(mw, infer_property_values=True)
    print("ДоказанноеУтверждение:", sorted(x.name for x in ДоказанноеУтверждение.instances()))
else:
    print("[запись] ДоказанноеУтверждение: ['C1', 'C2']")
mg = mw.as_rdflib_graph()
mshapes = rdflib.Graph(); mshapes.parse(data='''
@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix xsd: <http://www.w3.org/2001/XMLSchema#> . @prefix : <http://example.org/math.owl#> . @prefix shp: <http://example.org/shapes#> .
shp:Утверждение a sh:NodeShape ; sh:targetClass :Утверждение ;
  sh:property [ sh:path :доказывается ; sh:minCount 1 ; sh:message "у утверждения нет доказательства" ] ;
  sh:property [ sh:path ( :доказывается :вердикт ) ; sh:hasValue "принято"^^xsd:string ; sh:severity sh:Warning ; sh:message "вердикт внешней проверки не «принято»" ] .
''', format="turtle")
ok, rep, _ = проверить(mg, mshapes)
print("Conforms:", ok); print(rep[["узел", "сообщение", "строгость"]].to_string(index=False))

ДоказанноеУтверждение: ['C1', 'C2']
Conforms: False
узел                             сообщение строгость
  C3 вердикт внешней проверки не «принято»   Warning
  C4 вердикт внешней проверки не «принято»   Warning
  C4      у утверждения нет доказательства Violation


Класс отвечает «что следует»: C1 и C2 доказаны, C3 вне класса. Форма отвечает «что записано»: у C4 нет доказательства,
у C3 вердикт внешней проверки отличен от «принято». Математика здесь служит примером области с внешней проверкой.

## Что повторить самостоятельно

1. Запустить блокнот целиком. Прочитать отчёт проверки в полном виде.
2. Записать три компетентностных вопроса своей области и таблицу «вопрос — класс».
3. Добавить в свою онтологию выводимый класс по одному из вопросов, запустить машину вывода, зафиксировать результат до и после замыкания мира.
4. Написать три формы, прогнать проверку на чистых данных и на данных с намеренной ошибкой.
5. Записать результат в `kg/QUESTIONS.md`, `kg/VALIDATION.md` и `diary.md`.